# Question 3: xG vs Actual Conversion Rate by Shot Location

**Tier 1 — Foundational**

**Question:** Which shot locations produce the biggest gap between xG and actual conversion rate, and what does that tell us about shooter quality versus chance quality?

**Data:** StatsBomb Open Data + Understat

**Why this question matters:**  
xG is the most commonly cited metric in football analytics — and the most commonly misused. It measures the historical probability of a shot being scored based on its characteristics. It says nothing about who is taking the shot. This question forces you to confront that distinction directly: where does the model's prediction diverge most from reality, and what explains that divergence?

**What you are building toward:**
- A spatial map of xG vs actual conversion rate across the pitch
- Identification of zones where the model systematically over or underestimates
- A framework for distinguishing chance quality from finishing quality

---

**Critical lens (apply throughout):**
1. xG is a population-level model — what does that mean for evaluating individual players?
2. When would a player with low xG-per-shot still be a high-value attacker?
3. What shot characteristics does basic xG ignore that post-shot xG captures?

## Environment Setup

In [ ]:
# !pip install statsbombpy mplsoccer pandas numpy matplotlib seaborn networkx

In [2]:
# Import libraries
import numpy as np
import pandas as pd
from statsbombpy import sb
from mplsoccer import Pitch
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx


## Step 1: Load Shot Data from StatsBomb

Load shot events across multiple competitions to build a large enough sample.

**Things to consider:**
- More competitions = larger sample = more reliable zone-level estimates
- Filter to open play shots only — penalties and free kicks have different dynamics
- What columns do shot events contain in the StatsBomb schema?

In [4]:
# Load shot events from multiple StatsBomb competitions
# Filter to open play shots only
comps = sb.competitions()
comps = comps[comps['competition_id'].isin([9])]  # Filter to the competitions we want to load

# Get season ids and competition ids for the competitions we want to load
comp_season_pairs = comps[['competition_id', 'season_id']].drop_duplicates()

# Convert comp_season_pairs to a list of tuples for easier iteration
comp_season_tuples = list(comp_season_pairs.itertuples(index=False, name=None))

# Iterate over the competition-season pairs and load matches for each pair
matches = []
for competition_id, season_id in comp_season_tuples:
    season_matches = sb.matches(competition_id=competition_id, season_id=season_id)
    matches.append(season_matches) 
matches_df = pd.concat(matches, ignore_index=True)
matches_df.head()

# Load events for all matches
events = []
for match_id in matches_df['match_id']:
    match_events = sb.events(match_id=match_id)
    match_events['match_id'] = match_id  # Add match_id to each event for later merging
    events.append(match_events)
events_df = pd.concat(events, ignore_index=True)

# Filter to shots from open play
shots_df = events_df[
    (events_df['type'] == 'Shot') &
    (events_df['play_pattern'] == 'Regular Play')
].copy()

# Get most important columns for our analysis
shots_df = shots_df[['match_id', 'minute', 'second', 'team', 'player', 'location', 'shot_end_location', 'shot_outcome', 'shot_statsbomb_xg']]
shots_df.head()

,match_id,minute,second,team,player,location,shot_end_location,shot_outcome,shot_statsbomb_xg
4094,3895302,7,40,Bayer Leverkusen,Piero Martín Hincapié Reyna,"[114.6, 33.5]","[118.1, 35.7, 0.2]",Saved,0.143381
4097,3895302,16,0,Bayer Leverkusen,Granit Xhaka,"[89.2, 42.5]","[101.4, 41.3]",Blocked,0.021272
4099,3895302,18,28,Werder Bremen,Nick Woltemade,"[105.4, 45.1]","[106.1, 44.9]",Blocked,0.082293
4105,3895302,37,40,Bayer Leverkusen,Amine Adli,"[113.1, 27.5]","[120.0, 36.6, 2.8]",Post,0.074157
4108,3895302,50,35,Werder Bremen,Leonardo Bittencourt,"[97.5, 40.8]","[116.6, 40.5, 1.9]",Saved,0.048408


In [ ]:
# Explore the shot event schema
# What columns are available? Which ones are relevant?


## Step 2: Extract Shot Coordinates and Outcome

StatsBomb stores shot locations as coordinate pairs. Extract:
- x, y coordinates
- xG value (StatsBomb provides this)
- Shot outcome (goal or not goal)
- Shooter identity

**Note:** StatsBomb uses a 120x80 pitch coordinate system.

In [ ]:
# Extract shot coordinates, xG, and outcome


## Step 3: Bin the Pitch into Zones

Divide the attacking half of the pitch into zones. For each zone calculate:
- Total shots
- Total goals
- Mean xG per shot
- Actual conversion rate (goals / shots)
- Gap: conversion rate minus mean xG

**Design decision:** How many zones? Too few and you lose spatial nuance. Too many and individual zones have small samples. Document your choice and reasoning.

**My zone design decision:**

*(Document your binning approach and sample size reasoning here)*

In [ ]:
# Bin shots into pitch zones


In [ ]:
# Calculate xG vs actual conversion rate per zone


## Step 4: Visualise xG vs Conversion Rate

Build two pitch heatmaps side by side:
1. Mean xG per zone
2. Actual conversion rate per zone

Then build a third map showing the gap (conversion rate minus xG) — zones where the model overestimates in red, underestimates in blue.

**Use mplsoccer for all pitch visualisations.**

In [ ]:
# Build xG heatmap


In [ ]:
# Build actual conversion rate heatmap


In [ ]:
# Build gap map: conversion rate minus xG per zone


## Step 5: Identify Systematic Divergences

Which zones show the largest and most consistent gap between xG and conversion rate?

**Things to investigate:**
- Do central zones close to goal overperform xG? Why might that be?
- Do wide angles underperform? What does that suggest about the model?
- Are there zones where sample size is too small to trust the gap?

In [ ]:
# Identify top 5 zones by positive gap and top 5 by negative gap
# Flag zones with sample sizes below a threshold you define


## Step 6: Player-Level Analysis

For players with at least 50 open play shots in your dataset, calculate:
- Total xG
- Actual goals
- Goals minus xG (finishing performance above expectation)
- Conversion rate above expectation

**Critical question:** Is a player who consistently overperforms xG a better finisher — or are they taking shots from zones where the model systematically underestimates?

In [ ]:
# Calculate player-level finishing performance above expectation


In [ ]:
# Scatter plot: xG vs actual goals per player
# Label top overperformers and underperformers


## Step 7: Interpret and Critique Your Findings

**7a. Which zones show the biggest gap between xG and actual conversion rate — and what explains it?**

*(Write your answer here)*

**7b. Name a player in your dataset who overperforms xG consistently. Is that finishing quality or zone selection — and how do you tell the difference?**

*(Write your answer here)*

**7c. A scout uses xG to compare two strikers. Striker A has 0.15 xG per shot and scores 20% of attempts. Striker B has 0.12 xG per shot and scores 18% of attempts. Who is the better finisher? What additional information do you need?**

*(Write your answer here)*

**7d. What does post-shot xG capture that pre-shot xG does not — and when does that distinction matter most?**

*(Write your answer here)*

## Extension: Post-Shot xG Comparison

StatsBomb provides post-shot xG (PSxG) for some competitions. Where available, compare PSxG to pre-shot xG by zone. Does PSxG close the gap between model prediction and actual conversion rate?

In [ ]:
# Compare PSxG to xG where available


---

## Before You Move to Question 4

1. Could you explain the difference between xG and finishing quality to a manager in two sentences?
2. Where does your zone-level analysis break down — and does that affect your conclusions?
3. What is one question about striker evaluation this analysis cannot answer?

Add new questions to `OPEN_QUESTIONS.md` before moving on.